# AeroNetra — Model Training on VisDrone

**Purpose:** Fine-tune YOLOv8, YOLO11, and RT-DETR on the VisDrone vehicle detection dataset.

**Kaggle Setup:**
1. Add dataset: `aeronetra-visdrone-yolo` (output from notebook 01)
2. Accelerator: **GPU T4 x2** (or P100)
3. Internet: **ON** (needed to download pretrained weights on first run)

**Outputs:** Trained `.pt` weights saved to `/kaggle/working/` — download locally or save as Kaggle dataset.

---

### Training Strategy

| Model | Pretrained | Epochs | Image Size | Notes |
|-------|-----------|--------|------------|-------|
| YOLOv8n | yolov8n.pt | 50 | 640 | Fast baseline |
| YOLO11n | yolo11n.pt | 50 | 640 | Latest architecture |
| RT-DETR-l | rtdetr-l.pt | 30 | 640 | Transformer-based, no NMS |

In [ ]:
# ============================================================
# Cell 1: Install & verify environment
# ============================================================
!pip install -q ultralytics

import torch
import ultralytics

print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Ultralytics:  {ultralytics.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

if DEVICE == "cpu":
    print("\n⚠️  WARNING: No GPU detected. Training will be very slow.")
    print("   Enable GPU: Notebook Settings → Accelerator → GPU T4 x2")

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================
from pathlib import Path
import yaml

# --- EDIT THIS: path to the YOLO-format dataset from notebook 01 ---
# When you attach the notebook-01 output as a Kaggle input, the path is one of:
#   /kaggle/input/<dataset-slug>/visdrone_yolo/dataset.yaml
#   /kaggle/input/<dataset-slug>/dataset.yaml
DATASET_YAML_SRC = Path(
    "/kaggle/input/datasets/pratyush801/1-dataset-preparatio/visdrone_yolo/dataset.yaml"
)

# Output directory for trained models
OUTPUT_DIR = Path("/kaggle/working/trained_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared hyperparameters (from AeroNetra configs/inference/inference.yaml)
IMAGE_SIZE = 640
CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45
SEED = 42

print(f"Source YAML:   {DATASET_YAML_SRC}")
print(f"YAML exists:   {DATASET_YAML_SRC.exists()}")

if not DATASET_YAML_SRC.exists():
    # Fallback: try to auto-discover any dataset.yaml under /kaggle/input/
    candidates = list(Path("/kaggle/input").rglob("dataset.yaml"))
    print(f"\nAuto-discovery found {len(candidates)} dataset.yaml file(s):")
    for c in candidates:
        print(f"  {c}")
    if candidates:
        DATASET_YAML_SRC = candidates[0]
        print(f"\nUsing: {DATASET_YAML_SRC}")
    else:
        raise FileNotFoundError(
            "No dataset.yaml found. Attach the notebook-01 output as a Kaggle input."
        )

# ------------------------------------------------------------
# CRITICAL FIX: rewrite dataset.yaml so 'path' points to the actual
# mounted directory, not the /kaggle/working/... path baked in by notebook 01.
# The Kaggle /kaggle/input/ mount is read-only, so we write the patched
# yaml into /kaggle/working/.
# ------------------------------------------------------------
with open(DATASET_YAML_SRC) as f:
    ds_cfg = yaml.safe_load(f)

# The real dataset root is the directory containing dataset.yaml
real_root = DATASET_YAML_SRC.parent.resolve()
ds_cfg["path"] = str(real_root)

# Sanity-check that train/val image dirs exist under the real root
for split_key in ("train", "val"):
    split_rel = ds_cfg.get(split_key, "")
    if split_rel and not (real_root / split_rel).exists():
        raise FileNotFoundError(
            f"Split '{split_key}' points to '{split_rel}' but "
            f"{real_root / split_rel} does not exist."
        )

# Write patched yaml to a writable location
DATASET_YAML = Path("/kaggle/working/dataset.yaml")
with open(DATASET_YAML, "w") as f:
    yaml.dump(ds_cfg, f, default_flow_style=False, sort_keys=False)

print(f"\nPatched YAML: {DATASET_YAML}")
print(f"Dataset root: {real_root}")
print(f"Classes ({ds_cfg['nc']}): {ds_cfg['names']}")
print(f"Output dir:   {OUTPUT_DIR}")


## Train YOLOv8n

In [ ]:
# ============================================================
# Cell 3: Train YOLOv8n
# ============================================================
from ultralytics import YOLO
import time

print("="*60)
print("Training YOLOv8n on VisDrone")
print("="*60)

yolov8_model = YOLO("yolov8n.pt")  # Downloads pretrained COCO weights

t0 = time.time()
yolov8_results = yolov8_model.train(
    data=str(DATASET_YAML),
    epochs=50,
    imgsz=IMAGE_SIZE,
    batch=-1,           # Auto batch size based on GPU memory
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="yolov8n_visdrone",
    seed=SEED,
    patience=10,        # Early stopping
    save=True,
    save_period=10,     # Checkpoint every 10 epochs
    plots=True,
    verbose=True,
    exist_ok=True,
)
yolov8_time = time.time() - t0

print(f"\nYOLOv8n training completed in {yolov8_time/60:.1f} minutes")
print(f"Best weights: {OUTPUT_DIR / 'yolov8n_visdrone' / 'weights' / 'best.pt'}")

## Train YOLO11n

In [ ]:
# ============================================================
# Cell 4: Train YOLO11n
# ============================================================
from ultralytics import YOLO

print("="*60)
print("Training YOLO11n on VisDrone")
print("="*60)

yolo11_model = YOLO("yolo11n.pt")  # Downloads pretrained COCO weights

t0 = time.time()
yolo11_results = yolo11_model.train(
    data=str(DATASET_YAML),
    epochs=50,
    imgsz=IMAGE_SIZE,
    batch=-1,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="yolo11n_visdrone",
    seed=SEED,
    patience=10,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    exist_ok=True,
)
yolo11_time = time.time() - t0

print(f"\nYOLO11n training completed in {yolo11_time/60:.1f} minutes")
print(f"Best weights: {OUTPUT_DIR / 'yolo11n_visdrone' / 'weights' / 'best.pt'}")

## Train RT-DETR-l

In [ ]:
# ============================================================
# Cell 5: Train RT-DETR-l
# ============================================================
from ultralytics import RTDETR

print("="*60)
print("Training RT-DETR-l on VisDrone")
print("="*60)

rtdetr_model = RTDETR("rtdetr-l.pt")  # Downloads pretrained COCO weights

t0 = time.time()
rtdetr_results = rtdetr_model.train(
    data=str(DATASET_YAML),
    epochs=30,           # RT-DETR converges faster
    imgsz=IMAGE_SIZE,
    batch=-1,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="rtdetr_l_visdrone",
    seed=SEED,
    patience=8,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    exist_ok=True,
)
rtdetr_time = time.time() - t0

print(f"\nRT-DETR-l training completed in {rtdetr_time/60:.1f} minutes")
print(f"Best weights: {OUTPUT_DIR / 'rtdetr_l_visdrone' / 'weights' / 'best.pt'}")

In [ ]:
# ============================================================
# Cell 6: Training summary & weight collection
# ============================================================
import shutil

print("="*60)
print("TRAINING SUMMARY")
print("="*60)

models_trained = [
    ("YOLOv8n", "yolov8n_visdrone", yolov8_time),
    ("YOLO11n", "yolo11n_visdrone", yolo11_time),
    ("RT-DETR-l", "rtdetr_l_visdrone", rtdetr_time),
]

# Collect all best weights into a single directory for easy download
weights_dir = Path("/kaggle/working/best_weights")
weights_dir.mkdir(exist_ok=True)

for name, run_name, train_time in models_trained:
    best_pt = OUTPUT_DIR / run_name / "weights" / "best.pt"
    status = "OK" if best_pt.exists() else "MISSING"
    print(f"  {name:12s}  {train_time/60:6.1f} min  weights: {status}")
    
    if best_pt.exists():
        dst = weights_dir / f"{run_name}_best.pt"
        shutil.copy2(best_pt, dst)
        size_mb = dst.stat().st_size / (1024 * 1024)
        print(f"    → Copied to {dst.name} ({size_mb:.1f} MB)")

print(f"\nAll best weights collected in: {weights_dir}")
print("\nNext steps:")
print("  1. Download weights from /kaggle/working/best_weights/")
print("  2. OR: Save notebook output as Kaggle dataset 'aeronetra-trained-weights'")
print("  3. Place .pt files in your local outputs/models/ directory")
print("  4. Use them in local notebooks 03-06 via get_model_adapter()")

In [ ]:
# ============================================================
# Cell 7: Training curves (if plots were generated)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(24, 6))

for ax, (name, run_name, _) in zip(axes, models_trained):
    results_png = OUTPUT_DIR / run_name / "results.png"
    if results_png.exists():
        img = mpimg.imread(str(results_png))
        ax.imshow(img)
        ax.set_title(name, fontsize=14)
    else:
        ax.text(0.5, 0.5, f"{name}\n(no plot)", ha="center", va="center", fontsize=14)
    ax.axis("off")

plt.suptitle("Training Results — AeroNetra VisDrone", fontsize=16)
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves_overview.png", dpi=150)
plt.show()